# YTAgent — Free 3D Video + Kokoro Narration Worker

This notebook runs Wan 2.1 on Kaggle's free GPU. It connects OUTBOUND to YTAgent over HTTPS, so no Cloudflare tunnel or changing public URL is needed. Turn **Internet on** and select a **GPU** accelerator in Notebook settings. Keep the notebook running while a video is being generated.The worker uses Wan 2.1 for real 3D video scenes and Kokoro 82M for local documentary narration. Both run without paid generation APIs. Turn **Internet on** and select a **GPU** accelerator. Keep the notebook running while a film is being generated.

In [ ]:
import os, subprocess, sys, pathlib, importlib.util
repo_dir = "/kaggle/working/YTAgent"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/qingremzy/YTAgent.git", repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", "main"], check=True)
os.chdir(repo_dir + "/gpu-worker")
print("Checking Kaggle's preinstalled Wan GPU runtime without replacing PyTorch...")
import torch
missing = []
for module, package in [("ftfy", "ftfy"), ("imageio", "imageio[ffmpeg]"), ("soundfile", "soundfile>=0.12.1"), ("diffusers", "diffusers>=0.32.0"), ("transformers", "transformers>=4.45.0"), ("accelerate", "accelerate>=1.0.0"), ("huggingface_hub", "huggingface_hub"), ("safetensors", "safetensors")]:
    if importlib.util.find_spec(module) is None: missing.append(package)
if missing:
    print("Installing only missing packages (PyTorch will not be upgraded):", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)
try:
    from diffusers import AutoencoderKLWan, WanPipeline
    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
except Exception as exc:
    print("Updating Diffusers compatibility only:", str(exc)[:180])
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "diffusers>=0.32.0", "transformers>=4.45.0", "accelerate>=1.0.0"], check=True)
    from diffusers import AutoencoderKLWan, WanPipeline
    from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
import ftfy, imageio, soundfile
print("Wan runtime ready:", "torch", torch.__version__, "diffusers", __import__("diffusers").__version__)
print("Kokoro will be installed only after video scenes finish.")


In [ ]:
import os, torch
worker_token = os.environ.get("YT_AGENT_WORKER_TOKEN")
if not worker_token:
    from kaggle_secrets import UserSecretsClient
    worker_token = UserSecretsClient().get_secret("YT_AGENT_WORKER_TOKEN")
    os.environ["YT_AGENT_WORKER_TOKEN"] = worker_token
print("Private worker token loaded securely.")
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. Enable a Kaggle GPU accelerator.")

In [ ]:
import os, subprocess, sys
os.environ["YT_AGENT_SERVER_URL"] = "https://ytagent-production-1038.up.railway.app"
print("Starting one-job Kaggle GPU runner. It waits for the queued job, generates it, uploads scenes and narration, then exits.")
subprocess.run([sys.executable, "app.py", "--run-once", "--wait-seconds", "1200"], check=True)
print("YTAgent GPU job finished.")

In [ ]:
print("Kaggle batch worker cell completed. Check the YTAgent dashboard for the final video status.")

# Deprecated: Cloudflare tunnel is not used. The worker polls Railway over outbound HTTPS.

## How the free GPU worker runs

YTAgent launches this notebook as a Kaggle batch run only after a video job is queued. The worker claims one job, generates all ten Wan scenes and Kokoro narration tracks, uploads them to Railway, and exits. No always-on interactive session or Cloudflare tunnel is required.

Run cells 1–4 in order only when testing manually. In normal use, YTAgent starts the saved notebook automatically when the GPU job is ready.

In [ ]:
print("The one-job GPU runner reports its result to YTAgent. Check the studio's generation status for progress and completion.")

In [ ]:
import os, pathlib, torch
print("python_cwd=", os.getcwd())
print("repo_exists=", pathlib.Path('/kaggle/working/YTAgent').exists())
print("worker_dir_exists=", pathlib.Path('/kaggle/working/YTAgent/gpu-worker').exists())
print("app_exists=", pathlib.Path('/kaggle/working/YTAgent/gpu-worker/app.py').exists())
print("token_loaded=", bool(os.environ.get("YT_AGENT_WORKER_TOKEN")))
print("server_url_loaded=", bool(os.environ.get("YT_AGENT_SERVER_URL")))
print("gpu_available=", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu_name=", torch.cuda.get_device_name(0))